In [1]:
import sys
import yaml
import argparse
import numpy as np
import igraph as ig
import networkx as nx
import matplotlib.pyplot as plt
from qlgates.config import Config
from qlgates.run_dynamics import propagate_state, build_unitary, bell_state
from qlgates.qlgraphs import qldit, cart_qldit, compute_eigen_decomposition

In [2]:
print(sys.executable)   # should show your conda env path, not system python

"""Load YAML and merge into Config dataclass."""
with open("../configs/cartpdttest.yaml") as f:
    overrides = yaml.safe_load(f)  # plain dict from YAML

# These are computed in __post_init__, never let YAML set them
overrides.pop("l", None)
overrides.pop("lp", None)

# Create Config instance with merged parameters
cfg = Config(**overrides)

/Users/sahadebadrita/opt/anaconda3/envs/graphs/bin/python


In [3]:
print('Parameters for QL-bits:')
print(cfg.n, cfg.k, cfg.l, cfg.lp, cfg.d, cfg.coupling, cfg.periodic, cfg.full)
print('Parameters for dynamics:')
print(cfg.timesteps, cfg.deltat)

Parameters for QL-bits:
16 12 2 1 1 True False False
Parameters for dynamics:
2 0.1


In [4]:
#Create QL-resources -- QL-bits, Cartesian Products
qlbit_1 = qldit(cfg.n, cfg.k, cfg.l, cfg.d, cfg.coupling, cfg.periodic, cfg.full)
qlbit_2 = qldit(cfg.n, cfg.k, cfg.lp, cfg.d, cfg.coupling, cfg.periodic, cfg.full)
qlbit1_qlbit2 = cart_qldit(qlbit_1,qlbit_2)

In [5]:
e_qlbit1, v_qlbit1 = compute_eigen_decomposition(qlbit_1,top_k=2)
e_qlbit2, v_qlbit2 = compute_eigen_decomposition(qlbit_2,top_k=2)
e_qlbit1_qlbit2, v_qlbit1_qlbit2 = compute_eigen_decomposition(qlbit1_qlbit2,top_k=6)

In [6]:
print(e_qlbit1, e_qlbit2, e_qlbit1_qlbit2)

[14. 10.] [13. 11.] [27.         25.         23.         21.         16.22485517 15.92232062]


In [7]:
psi0_ = np.kron(v_qlbit1[:,0], v_qlbit1[:,0])  # ground state of the product system
psi0_ = np.kron(psi0_,v_qlbit1[:,0])  # ground state of the product system
psi0_ = psi0_ / np.linalg.norm(psi0_)  # normalize the state vector
print(psi0_.shape)

(32768,)


In [8]:
psi0 = v_qlbit1[:,0]  # ground state of the first QL-bit
for i in range(cfg.NQL - 1):
    psi0 = np.kron(psi0,v_qlbit1[:,0])  # ground state of the product system

psi0 = psi0 / np.linalg.norm(psi0)  # normalize the state vector
print(psi0.shape)

(1024,)


In [9]:
result = propagate_state(cfg,psi0,build_unitary)